# EEG — BIOT / IIIC **full fine-tune** (unfreeze the encoder), GPU

This notebook is the *full-data, full-fine-tune* evolution of `tools/colab_eeg.ipynb`.
The deployed EEG head was trained with the **encoder frozen** on a small CPU subset and
plateaus at **balanced accuracy 0.278** (κ ≈ 0.15). Here we **unfreeze the BIOT encoder**
and train end-to-end on a larger HMS subset on a GPU.

### Honest target — read before your defence
- **Baseline:** 0.278 balanced accuracy (frozen-encoder linear probe, 6-class IIIC).
- **Realistic target:** **0.45 – 0.55 balanced accuracy** — i.e. roughly BIOT's *published*
  IIIC level (~0.5). That is the ceiling we aim for, not 90 %.
- **Why 90 % is impossible here:** IIIC (Ictal–Interictal–Injury Continuum, 6 classes) is a
  genuinely ambiguous task — *expert neurologists disagree with each other* on these labels.
  The field scores it with **balanced accuracy** and **Cohen's κ** against *consensus* labels
  precisely because inter-rater agreement is far below 90 %. BIOT's own published full-data
  result is ≈ **0.5** balanced accuracy and that is considered strong. A model claiming 90 %
  on IIIC would be out-performing the experts who *made* the labels — a leakage red flag, not
  a result. The defensible claim after this notebook: *"the full GPU fine-tune lifts our
  screening head from 0.28 toward ≈ 0.5 balanced accuracy, matching the encoder authors'
  published level."*

### What you need first (one-time)
1. **Runtime → Change runtime type → T4 GPU.**
2. Build the lean repo zip on your PC and upload it to your **Google Drive root** (`My Drive/`):
   `python "Colab PFE/make_lean_zip.py"` → upload the produced `medical-platform.zip`.
   (It already contains the bundled BIOT encoder `EEG-PREST-16-channels.ckpt`.)
3. A **Kaggle API token** (`kaggle.json`) and a one-time acceptance of the
   [HMS competition rules](https://www.kaggle.com/competitions/hms-harmful-brain-activity-classification/rules).

### Colab timeout note (important)
A free **T4 session is capped at ~12 h** (and idles out sooner if you close the tab). This whole
run — rate-limited HMS download + full fine-tune + eval — fits comfortably for the default
`N_EEGS = 1500`. The **download cell is resilient and resumable**: it waits out Kaggle's rate
limit (HTTP 429) with a short cooldown instead of crashing, and re-running it skips parquet files
already on disk (and `train.csv` is cached), so a dropped session just continues. To train on more
data, raise `N_EEGS` across sessions and point `HMS_DIR` at a Drive folder so the downloaded data
survives a disconnect (slower I/O, but persistent).

Run the cells **top to bottom**. The final cell prints a `=== RESULTS ===` block — paste it into
`Colab PFE/RESULTS_TEMPLATE.md` and hand it back to Claude for local re-validation.

## 1 — Confirm the GPU
If this prints `NO GPU`, set **Runtime → Change runtime type → T4 GPU** and re-run. The
full fine-tune is impractically slow on CPU.

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 2 — Mount Drive and unpack the repo
Unzips `My Drive/medical-platform.zip` to `/content/medical-platform`. The flatten check (from
`tools/COLAB.md`) handles the case where the zip nested an extra top folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/medical-platform
!mkdir -p /content/medical-platform
!unzip -q "/content/drive/My Drive/medical-platform.zip" -d /content/medical-platform
import os
root = "/content/medical-platform"
if not os.path.exists(f"{root}/tools") and os.path.exists(f"{root}/medical-platform/tools"):
    root = f"{root}/medical-platform"
print("repo root:", root)
%cd $root

## 2b — Preflight: is the unzipped repo COMPLETE and FRESH?
The #1 cause of a failed run is a **stale `medical-platform.zip`** on Drive — one built
*before* `train_eeg_head.py` gained the `--unfreeze` flag this notebook needs. This cell
fails **loudly** (with exactly what to do) if any required file is missing or the trainer
is the old frozen-encoder version, instead of dying deep in section 7 with a cryptic error.

**If it fails:** rebuild the zip on your PC (`python "Colab PFE/make_lean_zip.py"`), replace
`My Drive/medical-platform.zip`, and re-run from section 2.

In [ ]:
# PREFLIGHT - verify the unzipped repo is COMPLETE and supports the full fine-tune.
# This is the cheapest insurance against the #1 failure: a STALE medical-platform.zip.
import os, sys
print("python:", sys.version.split()[0], "| torch:", __import__("torch").__version__)

REQUIRED = [
    "tools/train_eeg_head.py", "tools/eval_eeg.py", "tools/eeg_hms.py",
    "backend/apps/inference/biot/__init__.py",
    "backend/apps/inference/eeg_preprocess.py",
    "backend/apps/inference/model_loader.py",
    "backend/models_weights/biot/EEG-PREST-16-channels.ckpt",   # bundled BIOT encoder
]
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(root, f))]
if missing:
    raise RuntimeError(
        "STALE or INCOMPLETE zip - missing:\n  " + "\n  ".join(missing) +
        "\n\nRebuild + re-upload the lean zip from your PC:\n"
        "  python \"Colab PFE/make_lean_zip.py\"\n"
        "then replace My Drive/medical-platform.zip and re-run from section 2.")

# The full fine-tune REQUIRES --unfreeze; an old zip (frozen-encoder trainer) lacks it.
_tr = open(os.path.join(root, "tools/train_eeg_head.py"), encoding="utf-8").read()
if "--unfreeze" not in _tr:
    raise RuntimeError(
        "STALE zip: tools/train_eeg_head.py has NO --unfreeze flag - it is the older "
        "frozen-encoder trainer. Rebuild medical-platform.zip on your PC "
        "(python \"Colab PFE/make_lean_zip.py\"), re-upload to Drive, and re-run from section 2.")

_enc = os.path.join(root, "backend/models_weights/biot/EEG-PREST-16-channels.ckpt")
print("encoder ckpt present: {:.1f} MB".format(os.path.getsize(_enc) / 1e6))
print("PREFLIGHT OK - repo complete and supports --unfreeze. Proceed.")

## 3 — Install the extra deps
Colab already ships torch / numpy / pandas. BIOT needs `linear-attention-transformer`; the HMS
parquet reader needs `pyarrow`. (The full fine-tune needs nothing beyond what BIOT itself imports.)

In [ ]:
# Re-run this cell after EVERY runtime restart (pip installs are wiped; Drive + downloaded data survive).
# Colab already ships torch / numpy / pandas. BIOT needs linear-attention-transformer; the HMS parquet
# reader needs pyarrow. Nothing else is required for the full fine-tune.
!pip -q install linear-attention-transformer pyarrow
import sys
print("OK - deps installed on Python", sys.version.split()[0])

## 4 — Run configuration (edit here if you want)
`N_EEGS` is the only knob most people touch.

**Why this default and not "ALL".** HMS has ~**17,089** unique labelled EEGs. The download cell
fetches up to **N** of them **one file at a time** through the rate-limited Kaggle API (a 0.4 s
pause between files; on HTTP 429 it waits out a 90 s cooldown and retries, then skips a file only
if it stays stuck). Pulling all ~17 k in one session would take hours of throttled transfer and
risks the 12 h T4 cap — so there is no practical "ALL" switch. The default **`N_EEGS = 1500`**
reliably completes in a single session (~15–25 min including cooldowns) and gives several thousand
class-balanced windows — enough to reach the ~0.5 balanced-accuracy IIIC ceiling. `LIMIT` caps the
labelled windows actually used, balanced across the 6 classes, so RAM stays bounded. To use more
data, raise `N_EEGS`, point `HMS_DIR` at a Drive folder, and let the resumable download run across
sessions.

In [ ]:
import time

# --- main knobs ---
N_EEGS     = 1500          # unique HMS EEGs to use. If you pre-downloaded fewer, set this <= that
                           # count so the auto-detect uses your data instead of trying to download.
LIMIT      = 20000         # max labelled windows used, balanced across the 6 IIIC classes
EPOCHS     = 10            # --unfreeze wants FEW epochs (~6-12): the encoder is already pretrained
BATCH_SIZE = 32            # raw (16,2000) segments flow through the encoder; drop to 16 on OOM
ENC_LR     = "1e-5"        # encoder LR (small — fine-tune, don't wreck the pretraining)
HEAD_LR    = "1e-3"        # classification-head LR
SEED       = 0             # fixes BOTH the balanced sampling and the patient-disjoint split

# --- paths ---
HMS_DIR  = "data/hms"            # fallback only — the download cell AUTO-SEARCHES your Drive for a
                                 # folder with train.csv + train_eegs/ (e.g. "hms") and uses it, so
                                 # you usually don't need to set this. Fresh downloads land here.
OUT_CKPT = "/content/biot_iiic.pt"   # full BIOTClassifier state_dict (encoder + trained head)

T_START = time.time()
print("config:", dict(N_EEGS=N_EEGS, LIMIT=LIMIT, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE,
                       ENC_LR=ENC_LR, HEAD_LR=HEAD_LR, SEED=SEED, HMS_DIR=HMS_DIR, OUT_CKPT=OUT_CKPT))

## 5 — Kaggle credentials
**Skip this cell if your data is already in `HMS_DIR`** (pre-downloaded to a Drive folder) — no token
needed when nothing is downloaded. Otherwise:

First **accept the HMS rules once** (needed regardless of token type):
https://www.kaggle.com/competitions/hms-harmful-brain-activity-classification/rules

Two ways to authenticate — the cell below supports both:
- **New-style token (recommended)** — looks like `KGAT_…` (kaggle.com → *Settings* → *API* → create token). Just paste it when prompted; **no username, no kaggle.json needed**. The repo's downloader sends it as a Bearer token.
- **Legacy `kaggle.json`** — press Enter at the prompt instead, then upload the file.

The token is read with `getpass`, so it is never displayed or saved in the notebook.

In [ ]:
from getpass import getpass
import os

tok = getpass("Paste your Kaggle token (KGAT_...), or press Enter to upload kaggle.json instead: ").strip()
if tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
    print("KAGGLE_API_TOKEN set (Bearer auth - no username needed).")
else:
    from google.colab import files
    import json
    up = files.upload()                         # pick your kaggle.json
    k = json.load(open("kaggle.json"))
    os.environ["KAGGLE_USERNAME"] = k["username"]
    os.environ["KAGGLE_KEY"] = k["key"]
    print("legacy kaggle.json creds set for", k["username"])

## 6 — Get the HMS subset (auto-finds pre-downloaded data, else downloads)
This cell needs `train.csv` + a class-balanced set of `N_EEGS` per-EEG parquet files. It gets them
either way:

- **Already have the data in Drive?** It **auto-searches your Drive** for a folder containing
  `train.csv` + `train_eegs/` (a folder named `hms` / `hms_data`, or any folder with those) and uses
  it as-is — **no path to set, and you can skip the credentials cell (5)**. It only downloads if it
  finds nothing.
- **Otherwise** it downloads from Kaggle: **resilient** (waits out HTTP 429 with a cooldown, never
  crashes) and **resumable**. It only ever pulls the **`N_EEGS` subset (a few GB)** — never the full
  ~30 GB dataset.

> Make sure `N_EEGS` (config cell) is **≤ the number of parquet files you pre-downloaded**, so the
> auto-detect uses your data instead of trying to fetch more. Watch the printout: it shows the
> resolved `HMS_DIR =` and either *"using PRE-DOWNLOADED data"* or the download progress.

In [ ]:
# Get the HMS subset: AUTO-FIND pre-downloaded data in Drive, else download it.
# This ONLY ever fetches the N_EEGS subset (a few GB) - NEVER the full ~30 GB set.
# It first SEARCHES your Drive for a folder holding train.csv + train_eegs/ (e.g.
# "hms"); if found it uses that data as-is and skips Kaggle entirely (you can skip
# the credentials cell too). Otherwise it downloads, waiting out 429s and resuming.
import os, glob, time, sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, "tools")
import train_eeg_head as T   # light import (no torch at module level)

def _looks_like_hms(d):
    return bool(d) and os.path.exists(os.path.join(d, "train.csv")) and os.path.isdir(os.path.join(d, "train_eegs"))

def _find_hms(start):
    # 1) the configured path, 2) common Drive spots named hms / hms_data
    for c in [start,
              "/content/drive/MyDrive/hms", "/content/drive/My Drive/hms",
              "/content/drive/MyDrive/hms_data", "/content/drive/My Drive/hms_data"]:
        if _looks_like_hms(c):
            return c
    # 3) shallow search (<=2 levels) under Drive for any folder with the right files
    for base in ["/content/drive/MyDrive", "/content/drive/My Drive"]:
        if not os.path.isdir(base):
            continue
        for pat in ["train.csv", "*/train.csv", "*/*/train.csv"]:
            for csv in sorted(glob.glob(os.path.join(base, pat))):
                d = os.path.dirname(csv)
                if _looks_like_hms(d):
                    return d
    return start

HMS_DIR = _find_hms(HMS_DIR)   # may redirect to your Drive hms folder
HMS = Path(HMS_DIR)
(HMS / "train_eegs").mkdir(parents=True, exist_ok=True)
print("HMS_DIR =", HMS_DIR)

COOLDOWN = 120   # seconds to wait out a Kaggle 429 before retrying
MAX_WAITS = 10   # cooldowns to ride out per file (~20 min) before skipping it
n_have = len(list((HMS / "train_eegs").glob("*.parquet")))

# --- fast path: data already on disk (auto-found in Drive, or pre-downloaded) ---
if _looks_like_hms(HMS_DIR) and n_have >= N_EEGS:
    print("using PRE-DOWNLOADED data: {} EEG parquet files in {} (>= N_EEGS={}). No Kaggle download.".format(
        n_have, HMS, N_EEGS))

# --- otherwise download the class-balanced subset ---
else:
    def robust_get(fname, dest):
        """Download one file, waiting out 429 rate-limits. Returns True/False."""
        if dest.exists():
            return True
        for attempt in range(MAX_WAITS):
            try:
                T._kaggle_download_file(fname, dest)
                return True
            except Exception as e:
                if "429" in str(e) and attempt < MAX_WAITS - 1:
                    print("  429 on {} - cooling down {}s (try {}/{}) ...".format(
                        fname, COOLDOWN, attempt + 1, MAX_WAITS))
                    time.sleep(COOLDOWN); continue
                print("  FAILED {}: {}".format(fname, e))
                return False

    if not (HMS / "train.csv").exists():
        print("no pre-downloaded data found; downloading train.csv ...")
        if not robust_get("train.csv", HMS / "train.csv"):
            raise SystemExit(
                "Kaggle is hard rate-limiting this account (HTTP 429). Either WAIT ~30-60 min and re-run, "
                "OR put a pre-downloaded subset (train.csv + train_eegs/) in a Drive folder - this cell "
                "auto-finds it and skips the download.")

    df = pd.read_csv(HMS / "train.csv")
    total_eegs = df["eeg_id"].nunique()
    per = max(1, N_EEGS // df["expert_consensus"].nunique())
    chosen = (df.drop_duplicates("eeg_id")
                .groupby("expert_consensus", group_keys=False)
                .apply(lambda g: g.head(per)).head(N_EEGS))
    ids = chosen["eeg_id"].tolist()
    print("SUBSET: {} of {} unique EEGs (a few GB) - NOT the full ~30 GB dataset.".format(len(ids), total_eegs))

    missing = [e for e in ids if not (HMS / "train_eegs" / f"{e}.parquet").exists()]
    print("{}/{} present; downloading {} more (resilient + resumable) ...".format(len(ids) - len(missing), len(ids), len(missing)))
    t_dl = time.time(); done = len(ids) - len(missing); skipped = 0
    for i, eeg_id in enumerate(missing, 1):
        ok = robust_get("train_eegs/{}.parquet".format(eeg_id), HMS / "train_eegs" / "{}.parquet".format(eeg_id))
        done += int(ok); skipped += int(not ok)
        time.sleep(0.5)  # be polite between files
        if i % 100 == 0:
            print("  {}/{} new  ok={} skip={}  {:.0f} min".format(i, len(missing), done, skipped, (time.time() - t_dl) / 60))
    print("download wall-clock: {:.0f} min  (ok={} skip={})".format((time.time() - t_dl) / 60, done, skipped))

In [ ]:
# LOUD sanity check on the HMS subset BEFORE we spend GPU time training.
from pathlib import Path as _P
_hms = _P(HMS_DIR)
_n_parquet = len(list((_hms / "train_eegs").glob("*.parquet")))
if not (_hms / "train.csv").exists():
    raise RuntimeError("No train.csv in {} - download failed (Kaggle 429, or HMS rules not "
                       "accepted?). Accept the rules, then re-run the download cell.".format(_hms))
if _n_parquet == 0:
    raise RuntimeError(
        "0 EEG parquet files in {} - nothing to train on. Kaggle likely hard-throttled (HTTP 429): "
        "wait ~30-60 min and re-run the download cell, or drop a pre-downloaded subset "
        "(train.csv + train_eegs/) into a Drive folder (it is auto-found).".format(_hms / "train_eegs"))
if _n_parquet < N_EEGS:
    print("NOTE: only {} parquet files present (< N_EEGS={}). Training will use what is on disk.".format(
        _n_parquet, N_EEGS))
print("HMS sanity OK: train.csv present + {} EEG parquet files in {}".format(_n_parquet, _hms))

## 7 — Full fine-tune (unfreeze the encoder)
Runs `tools/train_eeg_head.py --unfreeze`: it builds `BIOTClassifier`, loads the bundled BIOT
encoder, **unfreezes it**, and trains encoder+head end-to-end with a class-balanced loss. It saves
the best-val-balanced-acc state to `OUT_CKPT` as a **full** `BIOTClassifier` state_dict — which
`model_loader.get_eeg_model()` loads as-is (drop it at
`backend/models_weights/biot/biot_iiic.pt` locally, no code change).

Watch the per-epoch `val balanced-acc`. If it is still climbing at the last epoch, raise `EPOCHS`.
On out-of-memory, lower `BATCH_SIZE` to 16 or `LIMIT` to 10000 in the config cell and re-run.

In [ ]:
import time, os, subprocess
t_train = time.time()
cmd = ["python", "tools/train_eeg_head.py", "--hms-dir", HMS_DIR, "--limit", str(LIMIT),
       "--unfreeze", "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
       "--encoder-lr", ENC_LR, "--lr", HEAD_LR, "--seed", str(SEED), "--out", OUT_CKPT]
print("running:", " ".join(cmd), "\n")
ret = subprocess.run(cmd)            # streams the live per-epoch printout to the cell
t_train_min = (time.time() - t_train) / 60
# LOUD FAIL: a bare "!python" would NOT stop the notebook on a non-zero exit, and cell 18
# would then torch.load a checkpoint that was never written. Check the exit code + output.
if ret.returncode != 0:
    raise RuntimeError(
        "TRAINING FAILED (exit {}). See the traceback above. Common causes: stale zip "
        "(rebuild medical-platform.zip), GPU OOM (lower BATCH_SIZE to 16 or LIMIT to 10000), "
        "or no data (re-run the download cell).".format(ret.returncode))
if not (os.path.exists(OUT_CKPT) and os.path.getsize(OUT_CKPT) > 0):
    raise RuntimeError("Trainer exited 0 but {} is missing/empty - aborting before eval.".format(OUT_CKPT))
print("\ntraining wall-clock: {:.0f} min  ->  {} ({:.1f} MB)".format(
    t_train_min, OUT_CKPT, os.path.getsize(OUT_CKPT) / 1e6))

## 8 — Evaluate on the patient-disjoint held-out split
Runs `tools/eval_eeg.py` with the **same `--limit` and `--seed`** as training, so it reproduces the
identical patient-disjoint split (no leakage). The printout (balanced-acc, κ, macro/weighted F1, KL,
per-class table, 6×6 confusion matrix) is also saved to `eeg_eval_report.txt`.

In [ ]:
import subprocess
cmd = ["python", "tools/eval_eeg.py", "--hms-dir", HMS_DIR, "--weights", OUT_CKPT,
       "--limit", str(LIMIT), "--seed", str(SEED)]
proc = subprocess.run(cmd, capture_output=True, text=True)   # eval is a small/fast split
report = proc.stdout + (("\n[stderr]\n" + proc.stderr) if proc.stderr.strip() else "")
print(report)
with open("eeg_eval_report.txt", "w", encoding="utf-8") as f:   # cell 9 copies this to Drive
    f.write(report)
if proc.returncode != 0:
    raise RuntimeError("EVAL FAILED (exit {}). See output above (often: missing checkpoint, "
                       "or no test windows in this split).".format(proc.returncode))

## 9 — Build a metrics `.json` and copy outputs to Drive
Recomputes the headline metrics by **reusing `eval_eeg.py`'s own functions** (so the numbers match
cell 8 exactly), writes `eeg_metrics.json`, and copies the checkpoint + json + text report to
`My Drive/colab_pfe_outputs/eeg/`. This runs inference once more on the (small) held-out split.

In [ ]:
import os, sys, json, shutil
import numpy as np
import torch

sys.path.insert(0, "tools")
sys.path.insert(0, "backend")
from apps.inference.biot import BIOTClassifier
from apps.inference.eeg_preprocess import IIIC_CLASSES
from eeg_hms import iter_segments, load_index, patient_split
from eval_eeg import _confusion, _metrics_from_confusion, _kl_divergence

assert os.path.exists(OUT_CKPT) and os.path.getsize(OUT_CKPT) > 0, \
    "checkpoint {} missing/empty - run the train cell (section 7) first.".format(OUT_CKPT)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BIOTClassifier(n_classes=6, n_channels=16, n_fft=200, hop_length=100)
state = torch.load(OUT_CKPT, map_location=device)
state = state.get("state_dict", state) if isinstance(state, dict) else state
model.load_state_dict(state)
model.eval().to(device)

# same patient-disjoint held-out split the trainer/eval used (same limit + seed)
samples = load_index(HMS_DIR, limit=LIMIT, balanced=True, seed=SEED)
_, test_s = patient_split(samples, test_frac=0.2, seed=SEED)

y_true, y_pred, votes, probs = [], [], [], []
with torch.no_grad():
    for s, seg in iter_segments(HMS_DIR, test_s):
        x = torch.from_numpy(seg[None].astype(np.float32)).to(device)
        p = torch.softmax(model(x), dim=1).cpu().numpy()[0]
        y_true.append(s["label"])
        y_pred.append(int(p.argmax()))
        votes.append(s["votes"])
        probs.append(p)

# LOUD FAIL: never silently report metrics on an empty / degenerate split.
if len(y_true) == 0:
    raise RuntimeError("0 held-out windows produced predictions - parquet files missing or the "
                       "split is empty. Re-run the download + train cells.")
if len(set(y_true)) < 2:
    print("WARNING: held-out set has a single class {} - metrics will be degenerate; "
          "raise N_EEGS / LIMIT.".format(sorted(set(y_true))))

c = _confusion(np.array(y_true), np.array(y_pred))
m = _metrics_from_confusion(c)
kl = _kl_divergence(votes, probs)

# Binary screening reframe (recall-first) — harmful pattern (any non-Other) vs Other.
# Mirrors tools/eval_eeg.py exactly, so these match the cell-8 report. For a SCREEN
# the clinically critical metric is "did a harmful window get flagged for review",
# not which exact IIIC type was named.
_OTHER = IIIC_CLASSES.index("Other") if "Other" in IIIC_CLASSES else 5
_SZ = IIIC_CLASSES.index("SZ") if "SZ" in IIIC_CLASSES else 0
_yt, _yp = np.array(y_true), np.array(y_pred)
_abn_t, _abn_p = _yt != _OTHER, _yp != _OTHER
_tp = int((_abn_t & _abn_p).sum()); _fn = int((_abn_t & ~_abn_p).sum())
_fp = int((~_abn_t & _abn_p).sum()); _tn = int((~_abn_t & ~_abn_p).sum())
_screen_recall = _tp / (_tp + _fn) if (_tp + _fn) else 0.0
_screen_prec = _tp / (_tp + _fp) if (_tp + _fp) else 0.0
_screen_spec = _tn / (_tn + _fp) if (_tn + _fp) else 0.0
_sz_mask = _yt == _SZ
_sz_caught, _sz_n = int((_sz_mask & _abn_p).sum()), int(_sz_mask.sum())
_sz_recall = _sz_caught / _sz_n if _sz_n else 0.0

metrics = {
    "run": "EEG BIOT/IIIC full fine-tune (unfreeze)",
    "baseline_balanced_acc": 0.278,
    "balanced_acc": round(float(m["balanced_acc"]), 4),
    "cohen_kappa": round(float(m["kappa"]), 4),
    "macro_f1": round(float(m["macro_f1"]), 4),
    "weighted_f1": round(float(m["weighted_f1"]), 4),
    "raw_accuracy": round(float(m["accuracy"]), 4),
    "kl_divergence": round(float(kl), 4),
    "n_test_windows": int(len(y_true)),
    # recall-first screening metrics (the clinically meaningful numbers for EEG)
    "binary_screen": {
        "abnormal_detection_recall": round(float(_screen_recall), 4),
        "abnormal_detection_precision": round(float(_screen_prec), 4),
        "benign_specificity": round(float(_screen_spec), 4),
        "seizure_routing_recall": round(float(_sz_recall), 4),
        "harmful_windows_missed": _fn,
        "seizure_caught_over_total": [_sz_caught, _sz_n],
    },
    "classes": list(IIIC_CLASSES),
    "per_class": {
        IIIC_CLASSES[i]: {
            "precision": round(float(m["precision"][i]), 4),
            "recall": round(float(m["recall"][i]), 4),
            "f1": round(float(m["f1"][i]), 4),
            "support": int(m["support"][i]),
        }
        for i in range(6)
    },
    "confusion_matrix": c.tolist(),
    "config": {
        "n_eegs": N_EEGS, "limit": LIMIT, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "encoder_lr": ENC_LR, "head_lr": HEAD_LR, "seed": SEED,
    },
}

DRIVE_OUT = "/content/drive/My Drive/colab_pfe_outputs/eeg"
os.makedirs(DRIVE_OUT, exist_ok=True)
DRIVE_CKPT = os.path.join(DRIVE_OUT, "biot_iiic.pt")
DRIVE_JSON = os.path.join(DRIVE_OUT, "eeg_metrics.json")
DRIVE_REPORT = os.path.join(DRIVE_OUT, "eeg_eval_report.txt")
shutil.copy(OUT_CKPT, DRIVE_CKPT)
with open(DRIVE_JSON, "w") as f:
    json.dump(metrics, f, indent=2)
if os.path.exists("eeg_eval_report.txt"):
    shutil.copy("eeg_eval_report.txt", DRIVE_REPORT)
print("copied checkpoint + metrics + report to", DRIVE_OUT)

## 10 — RESULTS (paste this whole block back to Claude)

In [ ]:
import time
print("=== RESULTS - paste this back to Claude ===")
print("run: EEG BIOT/IIIC FULL fine-tune (unfreeze encoder)")
print("baseline (frozen-encoder probe): balanced-acc 0.278  |  realistic target 0.45-0.55  |  BIOT published ~0.5")
print("config: N_EEGS={} LIMIT={} EPOCHS={} batch={} enc_lr={} head_lr={} seed={}".format(
    N_EEGS, LIMIT, EPOCHS, BATCH_SIZE, ENC_LR, HEAD_LR, SEED))
print("test windows (patient-disjoint): {}".format(metrics["n_test_windows"]))
print("balanced_acc : {:.3f}".format(metrics["balanced_acc"]))
print("cohen_kappa  : {:.3f}".format(metrics["cohen_kappa"]))
print("macro_f1     : {:.3f}".format(metrics["macro_f1"]))
print("weighted_f1  : {:.3f}".format(metrics["weighted_f1"]))
print("kl_divergence: {:.3f}  (true votes || predicted, lower is better)".format(metrics["kl_divergence"]))
print("raw_accuracy : {:.3f}  (misleading under imbalance)".format(metrics["raw_accuracy"]))
print("--- screening recall (recall-first; the clinically meaningful EEG numbers) ---")
_bs = metrics["binary_screen"]
print("abnormal_detection_recall : {:.3f}  ({} harmful windows missed as 'Other')".format(
    _bs["abnormal_detection_recall"], _bs["harmful_windows_missed"]))
print("abnormal_detection_prec   : {:.3f}".format(_bs["abnormal_detection_precision"]))
print("benign_specificity        : {:.3f}  (near 0 by design - it flags liberally)".format(_bs["benign_specificity"]))
print("seizure_routing_recall    : {:.3f}  ({}/{} seizures flagged for review)".format(
    _bs["seizure_routing_recall"], _bs["seizure_caught_over_total"][0], _bs["seizure_caught_over_total"][1]))
print("per-class F1:")
for name in metrics["classes"]:
    pc = metrics["per_class"][name]
    print("  {:<6} f1={:.3f} prec={:.3f} rec={:.3f} support={}".format(
        name, pc["f1"], pc["precision"], pc["recall"], pc["support"]))
print("confusion matrix (rows=true, cols=pred), class order {}:".format(metrics["classes"]))
for r in metrics["confusion_matrix"]:
    print("  {}".format(r))
print("files on Drive (My Drive/colab_pfe_outputs/eeg/):")
print("  checkpoint  : {}".format(DRIVE_CKPT))
print("  metrics json: {}".format(DRIVE_JSON))
print("  eval report : {}".format(DRIVE_REPORT))
print("place biot_iiic.pt at backend/models_weights/biot/biot_iiic.pt locally (no code change needed)")
print("runtime: training {:.0f} min  |  notebook total {:.0f} min".format(
    t_train_min, (time.time() - T_START) / 60))
print("=== END RESULTS ===")